# lp_audit: check a link-prediction evaluation for the errors in *The Protocol, Not the Model*

Four checks: **edge leakage**, **negative-sampling mismatch**, **model-free baseline / margin**, **dead candidate columns**.
It inspects the pairs you give it; it does not test whether a published number is inflated. Thresholds are heuristics.

In [ ]:
import os, sys, subprocess
REF = "feat/lp-audit"   # branch or tag of the repository to use
if not os.path.exists("lp_audit") and not os.path.exists("repo"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", REF,
                    "https://github.com/UlisesMoyaSanchez/microRNA_project.git", "repo"], check=True)
if os.path.exists("repo"):
    sys.path.insert(0, "repo")
import numpy as np
from lp_audit import audit

## 1. Synthetic examples
Each dataset is fictional and triggers one failure mode (`clean` triggers none). `model_auroc` is a made-up number.

In [ ]:
from lp_audit.examples.make_examples import build
for name, kw in build().items():
    print("=" * 100, "\n" + name)
    print(audit(**kw).render().split("\nScope:")[0])

## 2. Your own data
Pair files are `N x 2` arrays of `(row, col)` as `.npy` or CSV. Only `train_pos` and `heldout_pos` are required;
every extra input enables another check (otherwise it is reported as *unchecked*, never as *ok*).

In [ ]:
#@markdown Upload the files, then fill the names below (leave blank to skip an optional input).
try:
    from google.colab import files
    uploaded = files.upload()
except ImportError:
    pass   # not on Colab: put the files in the working directory

train_pos_file = "train_pos.npy"      #@param {type:"string"}
heldout_pos_file = "heldout_pos.npy"  #@param {type:"string"}
encoder_edges_file = ""               #@param {type:"string"}
train_neg_file = ""                   #@param {type:"string"}
eval_neg_file = ""                    #@param {type:"string"}
n_rows = 0                            #@param {type:"integer"}
n_cols = 0                            #@param {type:"integer"}
model_auroc = 0.0                     #@param {type:"number"}
mode = "bipartite"                    #@param ["bipartite", "homogeneous"]

from lp_audit import io as lio
opt = lambda f: lio.load_pairs(f) if f else None
report = audit(n_rows, n_cols, lio.load_pairs(train_pos_file), lio.load_pairs(heldout_pos_file),
               encoder_edges=opt(encoder_edges_file), train_neg=opt(train_neg_file),
               eval_neg=opt(eval_neg_file), model_auroc=model_auroc or None, mode=mode)
print(report.render())
report.save("audit.json")